# 01 · Dataset Construction and Leakage Audits

Starting point: the **5,505** articles left after duplicate cleaning (notebook 00).

This notebook

1. finds **related-article groups** (series / updated coverage of one event),
2. summarizes the **manual label audit** (250 articles, two reviewers),
3. loads and verifies the frozen **temporal split** (cut-off 26 Aug 2025),
4. runs a **cross-boundary similarity audit** between development and test,
5. assigns **5 group-aware stratified CV folds** to the development articles and
6. writes `data/article_index.csv`: the reproducibility record (URLs, dates, labels, split, group, fold, **no article text**).

> Raw CSV files with article text live in `data/raw/` and are not committed (see `data/README.md`).

In [1]:
import sys, re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import StratifiedGroupKFold
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

sys.path.append('..')
from src.preprocessing import normalize

RAW = '../data/raw/'
SEED = 42
CUTOFF = pd.Timestamp('2025-08-26')
CLASSES = ['sport', 'economy', 'health', 'arts', 'family']

clean = pd.read_csv(RAW + 'aljazeera_after_duplicate_cleaning.csv')
clean['publication_date'] = pd.to_datetime(clean['publication_date'])
print(clean.shape)
clean['section'].value_counts().reindex(CLASSES)

(5505, 8)


section
sport      1150
economy    1150
health     1148
arts       1150
family      907
Name: count, dtype: int64

## 1 · Related-article groups

Titles are represented with **character 3–5-gram TF-IDF**. Two articles are linked when their title cosine similarity is **≥ 0.75** and they were published **≤ 30 days** apart; connected articles form a candidate group.

In [2]:
X = TfidfVectorizer(analyzer='char', ngram_range=(3, 5)).fit_transform(clean['title'].fillna(''))
S = cosine_similarity(X, dense_output=False).tocoo()
days = np.abs((clean['publication_date'].values[S.row] - clean['publication_date'].values[S.col])
              .astype('timedelta64[D]').astype(int))
keep = (S.row < S.col) & (S.data >= 0.75) & (days <= 30)
A = csr_matrix((np.ones(keep.sum()), (S.row[keep], S.col[keep])), shape=(len(clean),) * 2)
_, comp = connected_components(A, directed=False)
clean['candidate_group'] = comp
size = clean['candidate_group'].map(clean['candidate_group'].value_counts())
cand = clean[size > 1].sort_values(['candidate_group', 'publication_date'])
print(f"Candidate groups: {cand['candidate_group'].nunique()}  |  articles: {len(cand)}\n")
for g, x in cand.groupby('candidate_group'):
    print(f'--- candidate {g} ({len(x)} articles)')
    for _, r in x.iterrows():
        print(f"   {r.section:8} {r.publication_date.date()}  {r.title[:70]}")

Candidate groups: 5  |  articles: 24

--- candidate 592 (4 articles)
   family   2024-10-28  الانتخابات الأميركية 2024
   arts     2024-10-30  الانتخابات الأميركية 2024
   family   2024-11-04  الانتخابات الأميركية 2024
   economy  2024-11-08  الانتخابات الأميركية 2024
--- candidate 935 (13 articles)
   arts     2026-02-12  رمضان 2026
   family   2026-02-17  رمضان 2026
   arts     2026-02-19  رمضان 2026
   family   2026-02-19  رمضان 2026
   family   2026-02-21  رمضان 2026
   family   2026-02-21  رمضان 2026
   health   2026-02-21  رمضان 2026
   health   2026-02-21  رمضان 2026
   health   2026-02-23  رمضان 2026
   economy  2026-03-05  رمضان 2026
   health   2026-03-06  رمضان 2026
   health   2026-03-11  رمضان 2026
   health   2026-03-18  رمضان 2026
--- candidate 1711 (3 articles)
   economy  2025-01-15  استقرار سعر صرف الليرة السورية أمام الدولار اليوم الأربعاء
   economy  2025-02-12  تراجع سعر صرف الليرة السورية أمام الدولار اليوم الأربعاء
   economy  2025-02-19  ارتفاع سعر صرف الليرة ال

### Manual review

Two candidate groups share an **identical generic title** (`رمضان 2026`, `الانتخابات الأميركية 2024`). Inspection showed these are a **scraping artifact**: for these pages the collector captured the topic tag instead of the headline. The articles cover unrelated stories, so these groups are **rejected**.

The remaining groups (Syrian-pound exchange-rate updates, and a "when to see a doctor" health series) are **confirmed** related-article series.

In [3]:
TOPIC_TAG_TITLES = {'رمضان 2026', 'الانتخابات الأميركية 2024'}   # manual decision

rejected = cand.groupby('candidate_group')['title'].apply(lambda t: set(t) <= TOPIC_TAG_TITLES)
confirmed_ids = rejected[~rejected].index
clean['group_id'] = None
for k, g in enumerate(confirmed_ids, start=1):
    clean.loc[clean['candidate_group'] == g, 'group_id'] = f'G{k}'

conf = clean[clean['group_id'].notna()]
print('Confirmed groups:', len(confirmed_ids), '| articles:', len(conf))
print('Articles with a topic tag instead of a headline:', clean['title'].isin(TOPIC_TAG_TITLES).sum())
conf[['group_id', 'section', 'publication_date', 'title']].sort_values(['group_id', 'publication_date'])

Confirmed groups: 3 | articles: 7
Articles with a topic tag instead of a headline: 19


,group_id,section,publication_date,title
1713,G1,economy,2025-01-15,استقرار سعر صرف الليرة السورية أمام الدولار ال...
1727,G1,economy,2025-02-12,تراجع سعر صرف الليرة السورية أمام الدولار اليو...
1731,G1,economy,2025-02-19,ارتفاع سعر صرف الليرة السورية مقابل الدولار ال...
1744,G2,economy,2025-03-26,ارتفاع سعر صرف الليرة السورية مقابل الدولار ال...
1768,G2,economy,2025-04-24,تراجع سعر صرف الليرة السورية مقابل الدولار الي...
4116,G3,health,2025-11-24,سعال الأطفال.. متى يستلزم استشارة الطبيب؟
4155,G3,health,2025-12-10,‫الأرق.. متى يستلزم استشارة الطبيب؟


## 2 · Label audit (250 articles, 50 per class)

In [4]:
audit = pd.read_csv(RAW + 'label_audit_final_results.csv')
r1, r2 = audit['reviewer1_label'], audit['reviewer2_label']
print(f"Reviewer agreement: {(r1 == r2).sum()}/{len(audit)} = {(r1 == r2).mean():.1%}")
print(f"Cohen's kappa:      {cohen_kappa_score(r1, r2):.3f}")

err = audit['source_label'] != audit['final_human_label']
print(f"Source label vs adjudicated human label: {err.sum()} disagreements ({err.mean():.1%})\n")
print((err.groupby(audit['source_label']).mean() * 100).reindex(CLASSES).rename('disagreement %'))
pd.crosstab(audit['source_label'], audit['final_human_label'],
            rownames=['Al Jazeera section'], colnames=['Human label']).reindex(index=CLASSES, columns=CLASSES)

Reviewer agreement: 223/250 = 89.2%
Cohen's kappa:      0.864
Source label vs adjudicated human label: 25 disagreements (10.0%)

source_label
sport       2.0
economy     2.0
health      4.0
arts        8.0
family     34.0
Name: disagreement %, dtype: float64


Human label,sport,economy,health,arts,family
Al Jazeera section,,,,,
sport,49,1,0,0,0
economy,0,49,1,0,0
health,0,1,48,0,1
arts,1,2,0,46,1
family,1,2,13,1,33


## 3 · Temporal split (frozen)

*Family* is the limiting class: after cleaning it has exactly **544** articles before and **363** on/after the cut-off. The same numbers were sampled from each other class, giving **907 per class** and **4,535** articles. The frozen selection is loaded and verified here.

In [5]:
final = pd.read_csv(RAW + 'aljazeera_final_balanced_4535.csv')
final['publication_date'] = pd.to_datetime(final['publication_date'])

fam = clean[clean['section'] == 'family']
print('Family before / after cut-off:', (fam.publication_date < CUTOFF).sum(), '/', (fam.publication_date >= CUTOFF).sum())

dev, test = final[final.final_split == 'development'], final[final.final_split == 'temporal_test']
assert dev.publication_date.max() < CUTOFF <= test.publication_date.min()
assert not set(dev.link) & set(test.link) and final.link.is_unique
print('Development:', len(dev), '| last date', dev.publication_date.max().date())
print('Test:       ', len(test), '| first date', test.publication_date.min().date())
pd.crosstab(final['section'], final['final_split']).reindex(CLASSES)

Family before / after cut-off: 544 / 363
Development: 2720 | last date 2025-08-25
Test:        1815 | first date 2025-08-26


final_split,development,temporal_test
section,,
sport,544,363
economy,544,363
health,544,363
arts,544,363
family,544,363


In [6]:
# Median publication date per class (development): Family is concentrated in later years
dev.groupby('section')['publication_date'].median().reindex(CLASSES)

section
sport     2023-02-26 00:00:00
economy   2023-09-30 12:00:00
health    2023-05-18 00:00:00
arts      2023-01-15 00:00:00
family    2024-11-13 12:00:00
Name: publication_date, dtype: datetime64[us]

## 4 · Cross-boundary similarity audit

Word 1–2-gram TF-IDF on normalized title + body, cosine similarity between **every development and every test article**.

In [7]:
text = (final['title'] + ' ' + final['content']).map(normalize)
V = TfidfVectorizer(ngram_range=(1, 2), min_df=1).fit_transform(text)
is_dev = (final.final_split == 'development').values
sim = cosine_similarity(V[is_dev], V[~is_dev])
i, j = np.unravel_index(sim.argmax(), sim.shape)
print(f'Max dev-test cosine similarity: {sim.max():.4f}')
print('Pairs >= 0.80:', int((sim >= 0.80).sum()))
print('Most similar pair:')
print('  dev :', final[is_dev].iloc[i][['section', 'publication_date', 'title']].tolist())
print('  test:', final[~is_dev].iloc[j][['section', 'publication_date', 'title']].tolist())

Max dev-test cosine similarity: 0.6188
Pairs >= 0.80: 0
Most similar pair:
  dev : ['economy', Timestamp('2025-03-26 00:00:00'), 'ارتفاع سعر صرف الليرة السورية مقابل الدولار اليوم الأربعاء']
  test: ['economy', Timestamp('2025-11-26 00:00:00'), 'استقرار سعر صرف الليرة السورية مقابل الدولار اليوم الأربعاء']


In [8]:
# Confirmed groups never cross the temporal boundary
g = final.merge(clean[['link', 'group_id']], on='link', how='left')
print(g.dropna(subset=['group_id']).groupby('group_id')['final_split'].agg(['unique', 'size']))

                   unique  size
group_id                       
G1          [development]     2
G2          [development]     2
G3        [temporal_test]     2


## 5 · Fixed cross-validation folds

`StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)` on the development portion. Articles of a confirmed group share a group id and therefore always fall in the same fold. **Every later experiment (Stage 1, Stage 2) reuses these folds.**

In [9]:
g['group_key'] = g['group_id'].fillna(g['link'])
g['fold'] = -1
dmask = g.final_split == 'development'
d = g[dmask]
for k, (_, va) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=SEED)
                            .split(d, d['section'], d['group_key'])):
    g.loc[d.index[va], 'fold'] = k
pd.crosstab(g.loc[dmask, 'fold'], g.loc[dmask, 'section']).reindex(columns=CLASSES)

section,sport,economy,health,arts,family
fold,,,,,
0,109,109,109,108,109
1,109,108,109,109,109
2,109,109,108,109,109
3,109,109,109,109,108
4,108,109,109,109,109


## 6 · Reproducibility record

In [10]:
g['split'] = g['final_split'].map({'development': 'development', 'temporal_test': 'test'})
g['title_is_topic_tag'] = g['title'].isin(TOPIC_TAG_TITLES)
index = g[['link', 'publication_date', 'section', 'split', 'group_id', 'fold', 'title_is_topic_tag']]
index = index.sort_values(['split', 'publication_date']).reset_index(drop=True)
index.to_csv('../data/article_index.csv', index=False)
print(index.shape)
index.head()

(4535, 7)


,link,publication_date,section,split,group_id,fold,title_is_topic_tag
0,https://www.aljazeera.net/family/2020/3/20/%d8...,2020-03-20,family,development,None,0,False
1,https://www.aljazeera.net/sport/2020/3/20/%d8%...,2020-03-20,sport,development,None,3,False
2,https://www.aljazeera.net/sport/2020/3/20/%d8%...,2020-03-20,sport,development,None,0,False
3,https://www.aljazeera.net/sport/2020/3/20/%d9%...,2020-03-20,sport,development,None,1,False
4,https://www.aljazeera.net/arts/2020/3/23/%d8%a...,2020-03-23,arts,development,None,1,False
